#### 공공 자전거 대여 데이터 파이프라인
**STEP 1 · 진단** — 1-1. 원본 그대로 읽기

In [1]:
import pandas as pd

stations_raw = pd.read_csv("data/stations.csv", dtype=str, keep_default_na=False)
bikes_raw    = pd.read_csv("data/raw-bikes.csv", dtype=str, keep_default_na=False)
rentals_raw  = pd.read_csv("data/raw-rentals.csv", dtype=str, keep_default_na=False)

print("stations :", stations_raw.shape)
print("bikes    :", bikes_raw.shape)
print("rentals  :", rentals_raw.shape)

stations : (30, 3)
bikes    : (55, 6)
rentals  : (15000, 8)


**1-2. 컬럼별 dtype 확인**

In [2]:
stations = pd.read_csv("data/stations.csv")
bikes    = pd.read_csv("data/raw-bikes.csv")
rentals  = pd.read_csv("data/raw-rentals.csv")

print("[stations]")
print(stations.dtypes)
print()
print("[bikes]")
print(bikes.dtypes)
print()
print("[rentals]")
print(rentals.dtypes)

[stations]
station_id      str
station_name    str
district        str
dtype: object

[bikes]
bike_id             str
station_id          str
bike_type           str
gear_count          str
manufacture_year    str
daily_fee           str
dtype: object

[rentals]
rental_id             str
bike_id               str
user_id               str
rent_time             str
return_time           str
distance_km       float64
fee                   str
payment_method        str
dtype: object


**1-3. 숫자여야 하는데 문자열로 읽힌 컬럼**


In [3]:
# 숫자여야 하는 컬럼만 골라서 dtype 확인
num_cols_bikes   = ["gear_count", "manufacture_year", "daily_fee"]
num_cols_rentals = ["distance_km", "fee"]

print("[bikes]")
print(bikes[num_cols_bikes].dtypes)
print()
print("[rentals]")
print(rentals[num_cols_rentals].dtypes)

[bikes]
gear_count          str
manufacture_year    str
daily_fee           str
dtype: object

[rentals]
distance_km    float64
fee                str
dtype: object


→ 문자열로 읽힌 숫자 컬럼: `gear_count`, `manufacture_year`, `daily_fee`, `fee` (4개)

→ `distance_km`은 float64지만, "N/A" 같은 값이 자동으로 NaN이 된 것이라 원본 확인 필요

**1-4. 컬럼별 결측 수와 비율**

In [4]:
missing = pd.DataFrame({
    "결측수": rentals.isna().sum(),
    "비율(%)": (rentals.isna().mean() * 100).round(2)
})
print("[rentals 결측]")
print(missing)

[rentals 결측]
                결측수  비율(%)
rental_id         0   0.00
bike_id           0   0.00
user_id           0   0.00
rent_time         0   0.00
return_time       0   0.00
distance_km     212   1.41
fee               0   0.00
payment_method    0   0.00


**1-5. distance_km 을 숫자로 못 바꾸는 값**

In [5]:
converted = pd.to_numeric(rentals_raw["distance_km"], errors="coerce")
bad = rentals_raw.loc[converted.isna(), "distance_km"]

print("변환 실패 건수:", len(bad))
print(bad.value_counts())

변환 실패 건수: 212
distance_km
N/A    212
Name: count, dtype: int64


→ distance_km 변환 실패값: "N/A" 212건 (기본 read_csv는 자동으로 NaN 처리 → float64로 읽힘)

**1-6. bike_id, rental_id 중복 건수**

In [6]:
print("bike_id 중복:", bikes_raw["bike_id"].duplicated().sum())
print("rental_id 중복:", rentals_raw["rental_id"].duplicated().sum())

bike_id 중복: 5
rental_id 중복: 500


→ bike_id 중복 5건 → 실제 자전거 50대 / rental_id 중복 500건 (배치 스크립트 오작동)

**1-7. bike_type, district, payment_method 고유값**

In [7]:
print("[bike_type]", bikes_raw["bike_type"].nunique(), "종류")
print(bikes_raw["bike_type"].unique())
print()
print("[district]", stations_raw["district"].nunique(), "종류")
print(stations_raw["district"].unique())
print()
print("[payment_method]", rentals_raw["payment_method"].nunique(), "종류")
print(rentals_raw["payment_method"].unique())

[bike_type] 6 종류
<StringArray>
['일반　', '일 반', '일반', '전동', 'electric', '일반형']
Length: 6, dtype: str

[district] 10 종류
<StringArray>
['강남구', '강동구', '강북구', '강서구', '관악구', '광진구', '구로구', '금천구', '노원구', '도봉구']
Length: 10, dtype: str

[payment_method] 13 종류
<StringArray>
[      'CARD',       ' APP', 'membership', 'Membership',       'APP ',
        'app',      'CARD ', 'MEMBERSHIP',        'App',      ' CARD',
       'card',       'Card',        'APP']
Length: 13, dtype: str


**1-8. bike_type 실제 종류 판단**

→ bike_type 고유값 6종 → 실제는 2종 (일반 / 전동)
  - 일반: '일반　'(전각 공백), '일 반', '일반', '일반형'
  - 전동: '전동', 'electric'

**1-9. bikes 숫자 컬럼 원본 값 확인**

In [8]:
for col in ["station_id","gear_count","manufacture_year", "daily_fee"]:
    print(f"[{col}]")
    print(bikes_raw[col].value_counts())
    print()

[station_id]
station_id
S23    4
S08    4
S13    3
S25    3
S03    3
S12    3
S02    3
s11    2
S05    2
S20    2
S30    2
S15    2
S24    2
S18    2
S06    2
S19    1
S26    1
s22    1
S10    1
S22    1
s23    1
s18    1
S28    1
S21    1
       1
s14    1
s26    1
S04    1
S27    1
S01    1
s03    1
Name: count, dtype: int64

[gear_count]
gear_count
3단    25
３단    20
7단     7
７단     3
Name: count, dtype: int64

[manufacture_year]
manufacture_year
2018    11
2016     8
2019     6
N/A      6
2015     6
2020     6
2023     5
2017     2
불명       2
2021     2
2022     1
Name: count, dtype: int64

[daily_fee]
daily_fee
1,000    45
2,000    10
Name: count, dtype: int64



**STEP 1 진단 결과 요약**

| 파일 | 컬럼 | 문제 | 건수 |
|---|---|---|---|
| bikes | bike_id | 중복 | 5 |
| bikes | station_id | 대소문자 혼재(S11/s11), 빈 값 | 빈 값 1 |
| bikes | bike_type | 6종 혼재(전각 공백, 가운데 공백, electric, 일반형) → 실제 2종 | - |
| bikes | gear_count | "단" 단위 포함, 전각 숫자(３단) | 전부 문자열 |
| bikes | manufacture_year | "N/A", "불명" | 8 |
| bikes | daily_fee | 콤마 포함(1,000) | 전부 문자열 |
| rentals | rental_id | 중복 | 500 |
| rentals | distance_km | "N/A" | 212 |
| rentals | fee | 콤마 포함(1,470) → 문자열로 읽힘 | - |
| rentals | payment_method | 앞뒤 공백, 대소문자 혼재 → 실제 3종 | 13종 |
| stations | - | 깨끗함 | - |

#### STEP 2 · 자전거 마스터 정제
**2-1. 전각 문자 → 반각으로 통일**

In [9]:
import unicodedata

def clean_text(s):
    s = unicodedata.normalize("NFKC", s)
    s = s.strip()
    return s

print(repr(clean_text("３단")))
print(repr(clean_text("일반　")))

'3단'
'일반'


**2-2. 전체 컬럼에 clean_text 적용**

In [10]:
bikes = bikes_raw.copy()

for col in bikes.columns:
    bikes[col] = bikes[col].map(clean_text)

print(bikes["gear_count"].value_counts())    

gear_count
3단    45
7단    10
Name: count, dtype: int64


**2-3. station_id — 대문자 통일, 빈 값은 결측**

In [11]:
bikes["station_id"] = bikes["station_id"].str.upper()
bikes["station_id"] = bikes["station_id"].replace("", pd.NA)

print("고유값 수:", bikes["station_id"].nunique())
print("결측 수:", bikes["station_id"].isna().sum())

고유값 수: 25
결측 수: 1


**2-4. bike_type — 일반 / 전동 두 가지로 통일**

In [12]:
bikes["bike_type"] = bikes["bike_type"].replace({
    "일 반":  "일반",
    "일반형": "일반",
    "electric": "전동",
})

print(bikes["bike_type"].unique())

<StringArray>
['일반', '전동']
Length: 2, dtype: str


**2-5. gear_count — 단위 문자 제거 후 정수로**

In [13]:
bikes["gear_count"] = bikes["gear_count"].str.replace("단","").astype(int)

print(bikes["gear_count"].dtype)
print(bikes["gear_count"].min(),"~",bikes["gear_count"].max())

int64
3 ~ 7


**2-6. daily_fee — 콤마 제거 후 정수로**

In [14]:
bikes["daily_fee"] = bikes["daily_fee"].str.replace(",","").astype(int)

print(bikes["daily_fee"].dtype)
print(bikes["daily_fee"].min(), "~", bikes["daily_fee"].max())

int64
1000 ~ 2000


**2-7. manufacture_year — 정수로 (변환 실패는 결측)**

In [15]:
bikes["manufacture_year"] = pd.to_numeric(bikes["manufacture_year"], errors="coerce").astype("Int64")

print(bikes["manufacture_year"].dtype)
print("결측 수:", bikes["manufacture_year"].isna().sum())
print(bikes["manufacture_year"].min(), "~", bikes["manufacture_year"].max())

Int64
결측 수: 8
2015 ~ 2023


**2-8. bike_id 기준 중복 제거**

In [16]:
print("제거 전:", len(bikes))

bikes = bikes.drop_duplicates(subset="bike_id")

print("제거 후:", len(bikes))

제거 전: 55
제거 후: 50


**2-9. STEP 2 결과 확인**

In [17]:
print("행 수:", len(bikes))
print("bike_type 고유값:", list(bikes["bike_type"].unique()))
print("station_id 고유값:", bikes["station_id"].nunique(), "종 + 결측", bikes["station_id"].isna().sum(), "건")
print("gear_count 범위:", bikes["gear_count"].min(), "~", bikes["gear_count"].max())
print("daily_fee 범위:", bikes["daily_fee"].min(), "~", bikes["daily_fee"].max())
print()
print(bikes.dtypes)

행 수: 50
bike_type 고유값: ['일반', '전동']
station_id 고유값: 25 종 + 결측 1 건
gear_count 범위: 3 ~ 7
daily_fee 범위: 1000 ~ 2000

bike_id               str
station_id            str
bike_type             str
gear_count          int64
manufacture_year    Int64
daily_fee           int64
dtype: object


#### STEP 3 · 대여 기록 정제
**3-1. fee — 콤마 제거 후 숫자로**

In [18]:
rentals = rentals_raw.copy()

rentals["fee"] = rentals["fee"].str.replace(",", "")
rentals["fee"] = pd.to_numeric(rentals["fee"], errors="coerce")

print(rentals["fee"].dtype)
print("fee 결측:", rentals["fee"].isna().sum())

int64
fee 결측: 0


**3-2. distance_km — 콤마 제거 후 숫자로 (변환 실패는 결측)**

In [19]:
rentals["distance_km"] = rentals["distance_km"].str.replace(",", "")
rentals["distance_km"] = pd.to_numeric(rentals["distance_km"], errors="coerce")

print(rentals["distance_km"].dtype)
print("distance_km 결측:", rentals["distance_km"].isna().sum())

float64
distance_km 결측: 212


**3-3. payment_method — 공백 제거 후 대문자로 통일**

In [20]:
rentals["payment_method"] = rentals["payment_method"].str.strip().str.upper()

print(rentals["payment_method"].nunique(), "종")
print(sorted(rentals["payment_method"].unique()))

3 종
['APP', 'CARD', 'MEMBERSHIP']


**3-4. rent_time, return_time — datetime으로 통일 (형식 3종)**

In [21]:
for col in ["rent_time", "return_time"]:
    rentals[col] = pd.to_datetime(rentals[col], format="mixed")

print(rentals[["rent_time", "return_time"]].dtypes)
print(rentals[["rent_time", "return_time"]].head())

rent_time      datetime64[us]
return_time    datetime64[us]
dtype: object
            rent_time         return_time
0 2022-04-18 20:28:00 2022-04-18 20:44:00
1 2022-09-24 09:36:00 2022-09-24 09:48:00
2 2022-05-19 09:53:00 2022-05-19 11:08:00
3 2022-09-07 16:20:00 2022-09-07 18:04:00
4 2022-10-11 06:39:00 2022-10-11 07:18:00


**3-5. rental_id 기준 중복 제거**

In [22]:
print("제거 전:", len(rentals))

rentals = rentals.drop_duplicates(subset="rental_id")

print("제거 후:", len(rentals))
print("distance_km 결측:", rentals["distance_km"].isna().sum())

제거 전: 15000
제거 후: 14500
distance_km 결측: 200


#### STEP 4 · 결합
**4-1. rentals + bikes (bike_id)**

In [23]:
merged = rentals.merge(
    bikes,
    on="bike_id",
    how="left",
    validate="many_to_one",
    indicator=True,
)

print("결합 전:", len(rentals))
print("결합 후:", len(merged))
print(merged["_merge"].value_counts())

결합 전: 14500
결합 후: 14500
_merge
both          14420
left_only        80
right_only        0
Name: count, dtype: int64


**4-2. 매칭 실패한 bike_id 확인**

In [24]:
failed = merged.loc[merged["_merge"] == "left_only", "bike_id"]

print("매칭 실패:", len(failed), "건")
print(failed.value_counts())

매칭 실패: 80 건
bike_id
B099    42
B100    38
Name: count, dtype: int64


**4-3. 매칭 실패 데이터의 의미와 처리**

→ 매칭 실패: B099(42건), B100(38건) = 80건. 자전거 마스터에 없는 bike_id

→ 고아 레코드(참조 무결성 위반). 분석에서 제외하고, ID와 건수를 기록해 운영팀에 마스터 보완 요청

**4-4. 매칭 실패 행 제외**

In [25]:
merged = merged.loc[merged["_merge"] == "both"]
merged = merged.drop(columns="_merge")

print("제외 후:", len(merged))

제외 후: 14420


**4-5. + stations (station_id)**

In [26]:
before = len(merged)

merged = merged.merge(
    stations_raw,
    on="station_id",
    how="left",
    validate="many_to_one",
)

print("결합 전:", before)
print("결합 후:", len(merged))
print("district 결측:", merged["district"].isna().sum())

결합 전: 14420
결합 후: 14420
district 결측: 304


**4-6. 자치구 결측 원인**

In [27]:
no_district = merged.loc[merged["district"].isna()]

print(no_district["bike_id"].value_counts())
print()
print(no_district[["bike_id", "station_id", "station_name", "district"]].head())

bike_id
B043    304
Name: count, dtype: int64

    bike_id station_id station_name district
1      B043        NaN          NaN      NaN
23     B043        NaN          NaN      NaN
122    B043        NaN          NaN      NaN
141    B043        NaN          NaN      NaN
216    B043        NaN          NaN      NaN


→ 자치구 결측 304건 = 모두 B043. 자전거 마스터에서 station_id가 빈 값(1건)이라 대여소 매칭 불가

→ 고아 레코드(B099, B100)와 달리 자전거는 존재하므로 행은 유지. 운영팀에 B043 배치 대여소 확인 요청

#### STEP 5 · 이상치와 논리 검사
**5-1. 파생 컬럼: duration_min (대여 시간, 분)**

In [28]:
diff = merged["return_time"] - merged["rent_time"]
merged["duration_min"] = diff.dt.total_seconds() / 60

print(merged[["rent_time", "return_time", "duration_min"]].head())
print()
print(merged["duration_min"].describe())

            rent_time         return_time  duration_min
0 2022-04-18 20:28:00 2022-04-18 20:44:00          16.0
1 2022-09-24 09:36:00 2022-09-24 09:48:00          12.0
2 2022-05-19 09:53:00 2022-05-19 11:08:00          75.0
3 2022-09-07 16:20:00 2022-09-07 18:04:00         104.0
4 2022-10-11 06:39:00 2022-10-11 07:18:00          39.0

count    14420.000000
mean        61.414147
std         33.633612
min        -29.000000
25%         32.000000
50%         61.000000
75%         91.000000
max        120.000000
Name: duration_min, dtype: float64


**5-2. 규칙 ① 반납 시각 ≤ 대여 시각**

In [29]:
rule1 = merged["duration_min"] <= 0

print("규칙 ① 탐지:", rule1.sum(), "건")
print(merged.loc[rule1, ["rent_time", "return_time", "duration_min"]].head())

규칙 ① 탐지: 50 건
               rent_time         return_time  duration_min
206  2022-02-19 11:00:00 2022-02-19 10:53:00          -7.0
425  2022-02-05 14:00:00 2022-02-05 13:45:00         -15.0
1041 2022-03-03 21:00:00 2022-03-03 20:36:00         -24.0
1943 2022-10-19 13:00:00 2022-10-19 12:31:00         -29.0
1973 2022-03-09 08:00:00 2022-03-09 07:31:00         -29.0


**5-3. 규칙 ② 요금 음수**

In [30]:
rule2 = merged["fee"] < 0

print("규칙 ② 탐지:", rule2.sum(), "건")
print(merged.loc[rule2, ["rental_id", "duration_min", "fee"]].head())

규칙 ② 탐지: 20 건
     rental_id  duration_min   fee
258    R014282          34.0  -424
1107   R014292          12.0 -1263
1647   R014284          33.0 -1720
1920   R014298          37.0  -905
1955   R014285          16.0  -527


**5-4. 규칙 ③ 평균 속도 > 50km/h (물리 상한)**

In [31]:
speed = merged["distance_km"] / (merged["duration_min"] / 60)   # km/h
rule3 = speed > 50

print("규칙 ③ 탐지:", rule3.sum(), "건")
print(speed.describe())

규칙 ③ 탐지: 650 건
count    14220.000000
mean        13.369050
std         19.658779
min        -81.000000
25%          3.881277
50%          7.475304
75%         14.129032
max        288.600000
dtype: float64


→ 분 ÷ 60 = 시간. 거리 ÷ 시간 = 시속(km/h). 50km/h **초과**만 이상치 (`>`)

→ distance_km 이 결측(NaN)인 행은 속도도 NaN → `NaN > 50` 은 False 라서 규칙 ③에 걸리지 않음 (결측은 STEP 6에서 처리)

**5-5. 규칙 합치기 (중복 제외) 후 제거**

In [32]:
any_rule = rule1 | rule2 | rule3      # 하나라도 걸리면 True (| = 또는)

summary = pd.DataFrame({
    "규칙": ["① 반납 ≤ 대여", "② 요금 음수", "③ 속도 > 50km/h", "합계 (중복 제외)"],
    "탐지 건수": [rule1.sum(), rule2.sum(), rule3.sum(), any_rule.sum()],
})
print(summary.to_string(index=False))

before = len(merged)
merged = merged.loc[~any_rule].copy()     # ~ = 반대(NOT) → 걸리지 않은 행만 남김
print()
print("제거 전:", before, "→ 제거 후:", len(merged))

           규칙  탐지 건수
    ① 반납 ≤ 대여     50
      ② 요금 음수     20
③ 속도 > 50km/h    650
   합계 (중복 제외)    719

제거 전: 14420 → 제거 후: 13701


→ 50 + 20 + 650 = 720 이지만 합계는 719. 규칙 ②와 ③에 **동시에** 걸린 행이 1건 있어서, `|` 로 합치면 한 번만 셈

→ 14,420 − 719 = **13,701**

**왜 IQR 대신 도메인 규칙을 썼는가**
- 전동(빠름)과 일반(느림) 자전거가 섞여 있어서, 일반 자전거의 부풀려진 거리값이 전체 분포에서는 전동의 정상값처럼 보일 수 있음 → IQR로는 못 잡음
- 반납이 대여보다 빠름, 요금 음수는 분포와 상관없이 **물리적·논리적으로 불가능** → 통계가 아니라 규칙으로 판정해야 함
- 속도 50km/h 상한은 "자전거로는 불가능하다"는 **도메인 지식**에서 나온 기준

#### STEP 6 · 결측 처리
**6-1. fee 결측 — 대여 시간 × 분당요금으로 복원**

In [33]:
fee_na = merged["fee"].isna()
print("fee 결측:", fee_na.sum(), "건")

# 분당요금 = 정상 행의 (요금 ÷ 대여 시간) 중앙값, 자전거 타입별
per_min = (merged.loc[~fee_na, "fee"] / merged.loc[~fee_na, "duration_min"]).groupby(merged.loc[~fee_na, "bike_type"]).median()
print("타입별 분당요금(중앙값):")
print(per_min.round(2))

restore = merged.loc[fee_na, "duration_min"] * merged.loc[fee_na, "bike_type"].map(per_min)
merged.loc[fee_na, "fee"] = restore.round()
print()
print("요금 복원:", fee_na.sum(), "건 / 복원 후 fee 결측:", merged["fee"].isna().sum())

fee 결측: 0 건
타입별 분당요금(중앙값):
bike_type
일반    22.00
전동    21.46
dtype: float64

요금 복원: 0 건 / 복원 후 fee 결측: 0


→ 이 데이터는 fee 결측이 0건이라 실제 복원은 0건. 그래도 결측이 생기면 위 코드가 채워 줌

**6-2. distance_km 결측 — 복원 불가 → 제외**

In [34]:
dist_na = merged["distance_km"].isna()
print("distance_km 결측:", dist_na.sum(), "건")

merged = merged.loc[~dist_na].copy()
print("최종 행 수:", len(merged))

distance_km 결측: 200 건
최종 행 수: 13501


**왜 요금은 복원하고 거리는 버렸는가**
- 요금은 **대여 시간 × 분당요금** 처럼 다른 컬럼으로 계산할 근거가 있음
- 이동 거리는 대여 시간이 같아도 사람마다 다르고, 계산할 근거가 없음 → 평균으로 채우면 **지어낸 값**이 됨
- 판단 기준: "채울 수 있는 근거가 있는가" (평균으로 때우는 것과 다름)

#### STEP 7 · 집계
**7-1. 전체 요약**

In [35]:
overall = {
    "행 수": len(merged),
    "자전거 수": merged["bike_id"].nunique(),
    "이용자 수": merged["user_id"].nunique(),
    "총 이동 거리(km)": round(merged["distance_km"].sum(), 1),
    "총 매출(원)": int(merged["fee"].sum()),
    "평균 대여 시간(분)": round(merged["duration_min"].mean(), 1),
}
for k, v in overall.items():
    print(f"{k:<12} {v:,}")

행 수          13,501
자전거 수        50
이용자 수        1,999
총 이동 거리(km)  102,524.7
총 매출(원)      18,979,110
평균 대여 시간(분)  64.6


**7-2. 자치구별 집계 (매출 내림차순)**

In [36]:
by_district = (
    merged.assign(district=merged["district"].fillna("(미상)"))   # B043 → (미상)
          .groupby("district")
          .agg(건수=("rental_id", "count"),
               이동거리=("distance_km", "sum"),
               매출=("fee", "sum"))
          .sort_values("매출", ascending=False)
)
by_district["이동거리"] = by_district["이동거리"].round(1)
print(by_district)

            건수     이동거리       매출
district                        
강북구       2993  22622.1  4180470
금천구       2125  16150.5  2978700
관악구       1961  14503.0  2745740
강동구       1586  12130.5  2248050
도봉구       1331  10157.6  1880620
광진구       1062   8092.8  1495470
강서구        817   6235.8  1152480
강남구        755   5782.5  1048310
노원구        321   2629.1   471350
(미상)       289   2180.4   409490
구로구        261   2040.4   368430


**7-3. 자전거 타입별 집계**

In [37]:
by_type = merged.groupby("bike_type").agg(
    건수=("rental_id", "count"),
    평균이동거리=("distance_km", "mean"),
    평균대여시간=("duration_min", "mean"),
).round(2)
print(by_type)

              건수  평균이동거리  평균대여시간
bike_type                       
일반         10833    7.60   64.56
전동          2668    7.56   64.95


**7-4. 결제수단별 건수**

In [38]:
print(merged["payment_method"].value_counts())

payment_method
MEMBERSHIP    4637
APP           4459
CARD          4405
Name: count, dtype: int64


**7-5. 검산 — 전동은 일반보다 짧은 시간에 더 긴 거리를 가야 함**

In [39]:
e, n = by_type.loc["전동"], by_type.loc["일반"]
print("전동 거리 > 일반 거리:", e["평균이동거리"] > n["평균이동거리"], f'({e["평균이동거리"]} vs {n["평균이동거리"]})')
print("전동 시간 < 일반 시간:", e["평균대여시간"] < n["평균대여시간"], f'({e["평균대여시간"]} vs {n["평균대여시간"]})')

전동 거리 > 일반 거리: False (7.56 vs 7.6)
전동 시간 < 일반 시간: False (64.95 vs 64.56)


→ **검산 결과: 성립하지 않음.** 전동/일반의 평균 거리·시간 차이가 1% 미만으로 사실상 같음

→ STEP 1~6의 모든 건수(14,500 / 80 / 304 / 50·20·650·719 / 200 / 13,501)는 기대 결과와 정확히 일치하므로 정제 단계 오류는 아님

→ bike_type 매핑(electric→전동, 일반형→일반)과 중복 행(B018 등 5건, 내용 동일)도 다시 확인함

→ 원본 데이터 자체에 타입별 차이가 없는 것으로 보임 → report.md 에 기록하고 강사님께 확인 요청

#### STEP 8 · DB 적재 (Oracle)
**8-1. 스키마 설계 — rental_log** (schema.sql 과 같은 내용)
- 금액·거리: `NUMBER` (FLOAT 금지 → 소수점 오차 방지)
- 날짜·시각: `DATE` (문자열 금지)
- ID: `VARCHAR2` (계산하지 않는 값)
- `rental_id` 에 PRIMARY KEY(= 유니크) → 재실행 안전성의 출발점
- 수업에서 Oracle을 쓰므로 MySQL의 `ON DUPLICATE KEY UPDATE` 대신 Oracle의 `MERGE INTO` 로 UPSERT

In [40]:
import os, time, logging
from decimal import Decimal
import oracledb
from dotenv import load_dotenv

load_dotenv("../10_database/.env")   # 접속 정보는 .env 에서만 읽음

def connect():
    return oracledb.connect(
        user=os.getenv("DB_USER"),
        password=os.getenv("DB_PASSWORD"),
        dsn=f'{os.getenv("DB_HOST", "127.0.0.1")}:{os.getenv("DB_PORT", "1521")}/{os.getenv("DB_NAME")}',
    )

# 로그: 화면 + 파일(pipeline.log) 동시 기록
logger = logging.getLogger("bikecity")
logger.setLevel(logging.INFO)
logger.handlers.clear()                      # 셀을 다시 실행해도 로그가 두 번 찍히지 않게
fmt = logging.Formatter("%(asctime)s [%(levelname)s] %(message)s", "%H:%M:%S")
for h in (logging.StreamHandler(), logging.FileHandler("pipeline.log", encoding="utf-8")):
    h.setFormatter(fmt)
    logger.addHandler(h)

conn = connect()
print("DB 연결 성공:", conn.version)
conn.close()

DB 연결 성공: 21.3.0.0.0


In [41]:
DDL = """
CREATE TABLE rental_log (
    rental_id       VARCHAR2(10)        NOT NULL,
    bike_id         VARCHAR2(10)        NOT NULL,
    user_id         VARCHAR2(10)        NOT NULL,
    rent_time       DATE                NOT NULL,
    return_time     DATE                NOT NULL,
    duration_min    NUMBER(5)           NOT NULL,
    distance_km     NUMBER(5,1)         NOT NULL,
    fee             NUMBER(8)           NOT NULL,
    payment_method  VARCHAR2(20)        NOT NULL,
    bike_type       VARCHAR2(10 CHAR)   NOT NULL,
    station_id      VARCHAR2(10),
    station_name    VARCHAR2(100 CHAR),
    district        VARCHAR2(30 CHAR),
    CONSTRAINT pk_rental_log PRIMARY KEY (rental_id)
)"""

conn = connect()
with conn.cursor() as cur:
    try:
        cur.execute(DDL)
        print("rental_log 테이블 생성")
    except oracledb.DatabaseError as e:
        if "ORA-00955" in str(e):            # 이미 있는 테이블 → 그대로 사용 (재실행 안전)
            print("rental_log 테이블이 이미 있음 → 그대로 사용")
        else:
            raise
conn.close()

rental_log 테이블이 이미 있음 → 그대로 사용


**8-2. 적재 — 청크 단위 UPSERT (MERGE INTO)**
- `executemany` 로 벌크 적재, `CHUNK_SIZE` 건마다 커밋
- 값이 **바뀐 행만** UPDATE (`WHERE` 절) → 같은 데이터를 다시 넣으면 갱신 0건
- 신규 = 적재 후 행 수 − 적재 전 행 수 / 갱신 = 실제 반영된 행 수 − 신규

In [42]:
COLS = ["rental_id", "bike_id", "user_id", "rent_time", "return_time", "duration_min",
        "distance_km", "fee", "payment_method", "bike_type", "station_id", "station_name", "district"]
KEY = "rental_id"
CHUNK_SIZE = 2000

using  = ", ".join(f":{i+1} AS {c}" for i, c in enumerate(COLS))
others = [c for c in COLS if c != KEY]
sets   = ", ".join(f"dst.{c} = src.{c}" for c in others)
# DECODE(a, b, 0, 1) : a와 b가 같으면 0 (NULL끼리도 같다고 봄), 다르면 1
changed = " OR ".join(f"DECODE(dst.{c}, src.{c}, 0, 1) = 1" for c in others)

UPSERT = f"""
MERGE INTO rental_log dst
USING (SELECT {using} FROM dual) src
ON (dst.{KEY} = src.{KEY})
WHEN MATCHED THEN
    UPDATE SET {sets}
    WHERE {changed}
WHEN NOT MATCHED THEN
    INSERT ({", ".join(COLS)})
    VALUES ({", ".join("src." + c for c in COLS)})
"""

def prepare_rows(df):
    """DataFrame → list[tuple]. 타입을 DB 컬럼에 맞추고 NaN → None"""
    data = df[COLS].copy()
    data["duration_min"] = data["duration_min"].round().astype(int)
    data["fee"]          = data["fee"].round().astype(int)
    data = data.astype(object).where(data.notna(), None)
    # 거리: float 대신 Decimal 로 보내야 DB의 NUMBER(5,1) 과 값이 정확히 같음 (소수점 오차 방지)
    data["distance_km"] = [Decimal(str(round(v, 1))) for v in data["distance_km"]]
    rows = []
    for r in data.itertuples(index=False):
        # 시각: pandas Timestamp → 파이썬 datetime (DB의 DATE 로 바인딩)
        rows.append(tuple(v.to_pydatetime() if isinstance(v, pd.Timestamp) else v for v in r))
    return rows

def row_count(conn):
    with conn.cursor() as cur:
        cur.execute("SELECT COUNT(*) FROM rental_log")
        return cur.fetchone()[0]

def load(df, run_no):
    rows = prepare_rows(df)
    conn = connect()
    before = row_count(conn)
    affected = 0
    start = time.perf_counter()
    try:
        for i in range(0, len(rows), CHUNK_SIZE):
            part = rows[i:i + CHUNK_SIZE]
            with conn.cursor() as cur:
                cur.executemany(UPSERT, part)
                affected += cur.rowcount
            conn.commit()                                    # 청크 단위 커밋
            logger.info(f"  [{run_no}회차] 청크 {i:>5} ~ {i + len(part) - 1:>5} 커밋")
        after = row_count(conn)
    except Exception:
        conn.rollback()
        logger.exception(f"  [{run_no}회차] 적재 실패 (청크 시작 {i})")
        raise
    finally:
        conn.close()

    inserted = after - before
    updated  = affected - inserted
    logger.info(f"[{run_no}회차] 신규 {inserted}건 / 갱신 {updated}건 / 최종 {after}행 ({time.perf_counter() - start:.1f}초)")
    return inserted, updated, after

**8-3. 재실행 검증 — 같은 적재를 두 번 실행**

In [43]:
results = {}
for run_no in (1, 2):
    results[f"{run_no}회차"] = load(merged, run_no)

print(pd.DataFrame(results, index=["신규", "갱신", "최종 행 수"]))

16:05:28 [INFO]   [1회차] 청크     0 ~  1999 커밋
16:05:28 [INFO]   [1회차] 청크  2000 ~  3999 커밋
16:05:28 [INFO]   [1회차] 청크  4000 ~  5999 커밋
16:05:28 [INFO]   [1회차] 청크  6000 ~  7999 커밋
16:05:28 [INFO]   [1회차] 청크  8000 ~  9999 커밋
16:05:28 [INFO]   [1회차] 청크 10000 ~ 11999 커밋
16:05:28 [INFO]   [1회차] 청크 12000 ~ 13500 커밋
16:05:28 [INFO] [1회차] 신규 0건 / 갱신 0건 / 최종 13501행 (0.1초)
16:05:28 [INFO]   [2회차] 청크     0 ~  1999 커밋
16:05:28 [INFO]   [2회차] 청크  2000 ~  3999 커밋
16:05:28 [INFO]   [2회차] 청크  4000 ~  5999 커밋
16:05:28 [INFO]   [2회차] 청크  6000 ~  7999 커밋
16:05:28 [INFO]   [2회차] 청크  8000 ~  9999 커밋
16:05:28 [INFO]   [2회차] 청크 10000 ~ 11999 커밋
16:05:28 [INFO]   [2회차] 청크 12000 ~ 13500 커밋
16:05:28 [INFO] [2회차] 신규 0건 / 갱신 0건 / 최종 13501행 (0.0초)


          1회차    2회차
신규          0      0
갱신          0      0
최종 행 수  13501  13501


→ 기대 결과: 1회차 신규 13,501 / 2회차 신규 0, 갱신 0 / 최종 행 수 13,501 (두 번 모두)

→ 2회차에 행이 두 배가 되지 않는 이유: rental_id PRIMARY KEY + MERGE(UPSERT)

**8-4. 적재 검증 — 집계값 대조 (DataFrame vs DB)**

In [44]:
conn = connect()
with conn.cursor() as cur:
    cur.execute("""
        SELECT COUNT(*), COUNT(DISTINCT bike_id), COUNT(DISTINCT user_id),
               SUM(fee), SUM(distance_km), MIN(rent_time), MAX(rent_time)
        FROM rental_log
    """)
    n, bikes_n, users_n, fee_sum, dist_sum, min_t, max_t = cur.fetchone()
conn.close()

checks = {
    "행 수":        (len(merged),                         n),
    "자전거 수":    (merged["bike_id"].nunique(),          bikes_n),
    "이용자 수":    (merged["user_id"].nunique(),          users_n),
    "총 매출":      (int(merged["fee"].round().sum()),      int(fee_sum)),
    "총 이동 거리": (round(merged["distance_km"].round(1).sum(), 1), round(float(dist_sum), 1)),
    "최소 날짜":    (str(merged["rent_time"].min()),       str(min_t)),
    "최대 날짜":    (str(merged["rent_time"].max()),       str(max_t)),
}

all_ok = True
for name, (exp, act) in checks.items():
    ok = str(exp) == str(act)
    all_ok &= ok
    logger.info(f"  {'OK  ' if ok else 'FAIL'} {name:<8} DataFrame={exp} / DB={act}")
print("검증 결과:", "모두 일치" if all_ok else "불일치 있음")

16:05:28 [INFO]   OK   행 수      DataFrame=13501 / DB=13501
16:05:28 [INFO]   OK   자전거 수    DataFrame=50 / DB=50
16:05:28 [INFO]   OK   이용자 수    DataFrame=1999 / DB=1999
16:05:28 [INFO]   OK   총 매출     DataFrame=18979110 / DB=18979110
16:05:28 [INFO]   OK   총 이동 거리  DataFrame=102524.7 / DB=102524.7
16:05:28 [INFO]   OK   최소 날짜    DataFrame=2022-01-01 07:26:00 / DB=2022-01-01 07:26:00
16:05:28 [INFO]   OK   최대 날짜    DataFrame=2022-12-31 22:52:00 / DB=2022-12-31 22:52:00


검증 결과: 모두 일치


→ 합계가 미세하게 다르면: 파이썬 쪽 반올림(round(1))과 DB 컬럼 자릿수 NUMBER(5,1)가 맞는지 확인

In [45]:
merged.to_csv("data/clean_rentals.csv", index=False, encoding="utf-8-sig")
print(len(merged))

13501
